# Ensemble Methods Example (Wine Quality Dataset)

Here it is demonstrated how to use the `RandomForestClassifier`, `RandomForestRegressor`, and `GradientBoostingRegressor` modules from the CMOR-438 library.
In this example, the Wine Quality dataset is used to train, test, and evaluate all three ensemble models.

**Goal: Show how combining many decision trees improves accuracy over a single tree.**

## 1. Setup and Data Loading

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sys, os

# Add the algorithm folder and shared helpers to path
NOTEBOOK_DIR = os.path.abspath('')
sys.path.insert(0, NOTEBOOK_DIR)
sys.path.insert(0, os.path.join(NOTEBOOK_DIR, '..', '_shared'))

# Data lives two levels up from the algorithm folder
DATA_DIR = os.path.join(NOTEBOOK_DIR, '..', '..', 'data')
from ensemble_methods import RandomForestClassifier, RandomForestRegressor, GradientBoostingRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

wine = pd.read_csv(os.path.join(DATA_DIR, 'WineQT.csv')).drop(columns=['Id'])
FEATURE_COLS = [c for c in wine.columns if c != 'quality']
print(f"Dataset loaded: {wine.shape[0]} samples, {len(FEATURE_COLS)} features.")

## 2. Preprocessing

In [ ]:
X = StandardScaler().fit_transform(wine[FEATURE_COLS].values.astype(float))
y_reg = wine['quality'].values.astype(float)
y_clf = np.array([0 if q<=4 else (1 if q<=6 else 2) for q in y_reg])

X_tr, X_te, y_tr_clf, y_te_clf = train_test_split(X, y_clf, test_size=0.2, random_state=42, stratify=y_clf)
_, __, y_tr_reg, y_te_reg = train_test_split(X, y_reg, test_size=0.2, random_state=42)

print(f"Training samples: {X_tr.shape[0]}  |  Test samples: {X_te.shape[0]}")

## 3. Train All Three Models

In [ ]:
rf_c = RandomForestClassifier(n_estimators=80, max_depth=8, random_state=42).fit(X_tr, y_tr_clf)
rf_r = RandomForestRegressor(n_estimators=80, max_depth=6, random_state=42).fit(X_tr, y_tr_reg)
gbr  = GradientBoostingRegressor(n_estimators=150, learning_rate=0.08, max_depth=4, random_state=42).fit(X_tr, y_tr_reg)

print(f'Random Forest Classifier Accuracy: {rf_c.accuracy(X_te, y_te_clf):.4f}')
print(f'Random Forest Regressor R²:        {rf_r.score(X_te, y_te_reg):.4f}')
print(f'Gradient Boosting R²:              {gbr.score(X_te, y_te_reg):.4f}')

## 4. Results and Visualisation

Three plots are produced:
- **Feature importances** — averaged across all 80 trees; more stable than a single tree
- **RF Regressor predicted vs actual** — scatter of test predictions
- **Gradient Boosting training loss** — residual MSE per boosting round

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

idx = np.argsort(rf_c.feature_importances_)[::-1]
colors = plt.cm.plasma(np.linspace(0.15, 0.85, len(FEATURE_COLS)))
axes[0].bar(range(len(FEATURE_COLS)), rf_c.feature_importances_[idx], color=colors, edgecolor='white')
axes[0].set_xticks(range(len(FEATURE_COLS)))
axes[0].set_xticklabels([FEATURE_COLS[i] for i in idx], rotation=40, ha='right', fontsize=8)
axes[0].set_title('Random Forest - Feature Importances', fontweight='bold')
axes[0].set_ylabel('Importance')

preds_rf = rf_r.predict(X_te)
axes[1].scatter(y_te_reg, preds_rf, alpha=0.4, s=18, color='steelblue')
lo, hi = y_te_reg.min()-0.2, y_te_reg.max()+0.2
axes[1].plot([lo,hi],[lo,hi],'r--',lw=1.5)
axes[1].set_xlabel('Actual Quality'); axes[1].set_ylabel('Predicted Quality')
axes[1].set_title(f'Random Forest Regressor  R²={rf_r.score(X_te,y_te_reg):.3f}', fontweight='bold')

axes[2].plot(gbr.train_loss_, color='darkorange', lw=1.5)
axes[2].set_xlabel('Boosting Round'); axes[2].set_ylabel('MSE (residuals)')
axes[2].set_title('Gradient Boosting - Training Loss', fontweight='bold')
plt.tight_layout(); plt.show()